In [2]:
import re
import json


def hex_to_rgb(hex_color: str):
    """#RRGGBB → [R, G, B]"""
    hex_color = hex_color.lstrip("#")
    return [int(hex_color[i:i+2], 16) for i in (0, 2, 4)]


def parse_html_to_colors(html: str):
    """
    从 HTML 源码中解析 Artkal 颜色
    """
    pattern = re.findall(
        r'text-lg">([A-Z0-9]+)</div>.*?#([0-9A-Fa-f]{6})',
        html,
        re.S
    )

    colors = []
    seen = set()

    for code, hex_color in pattern:
        # 去重（防止重复解析）
        if code in seen:
            continue
        seen.add(code)

        hex_color = "#" + hex_color.upper()

        colors.append({
            "code": code,
            "hex": hex_color,
            "rgb": hex_to_rgb(hex_color)
        })

    # 排序（A1, A2, A10 这种自然排序）
    def sort_key(item):
        prefix = ''.join(filter(str.isalpha, item["code"]))
        number = int(''.join(filter(str.isdigit, item["code"])))
        return (prefix, number)

    colors.sort(key=sort_key)

    return colors


def save_to_json(data, output_path="artkal_colors.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    # 👉 把你扒下来的 HTML 粘到这里
    lst = ["A", "B", "C", "D", "E", "F", "G", "H", "M"]
    C = []
    for letter in lst:
        print(f"正在解析 {letter} 系列颜色...")
        with open(f"input_{letter}.html", "r", encoding="utf-8") as f:
            html = f.read()

        colors = parse_html_to_colors(html)
        C.extend(colors)

    save_to_json(C)

    print(f"✅ 已生成 {len(C)} 个颜色 -> artkal_colors.json")

正在解析 A 系列颜色...
正在解析 B 系列颜色...
正在解析 C 系列颜色...
正在解析 D 系列颜色...
正在解析 E 系列颜色...
正在解析 F 系列颜色...
正在解析 G 系列颜色...
正在解析 H 系列颜色...
正在解析 M 系列颜色...
✅ 已生成 221 个颜色 -> artkal_colors.json


In [3]:
import re
import json


def hex_to_rgb(hex_color: str):
    """#RRGGBB → [R, G, B]"""
    hex_color = hex_color.lstrip("#")
    return [int(hex_color[i:i+2], 16) for i in (0, 2, 4)]


def parse_html_to_colors(html: str):
    """
    从 HTML 源码中解析 Artkal 颜色（稳定版）
    """
    # ✅ 用 background-color + code 双锚点，避免串匹配
    pattern = re.findall(
        r'background-color:#([0-9A-Fa-f]{6}).*?text-lg">([A-Z0-9]+)</div>',
        html,
        re.S
    )

    colors = []
    seen = set()

    for hex_color, code in pattern:
        # 去重
        if code in seen:
            continue
        seen.add(code)

        hex_color = "#" + hex_color.upper()

        colors.append({
            "code": code,
            "hex": hex_color,
            "rgb": hex_to_rgb(hex_color)
        })

    # ✅ 更安全的自然排序
    def sort_key(item):
        prefix = ''.join(filter(str.isalpha, item["code"]))
        num_part = ''.join(filter(str.isdigit, item["code"]))
        number = int(num_part) if num_part else 0
        return (prefix, number)

    colors.sort(key=sort_key)

    return colors


def save_to_json(data, output_path="artkal_colors.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    # 👉 HTML 文件列表
    lst = ["A", "B", "C", "D", "E", "F", "G", "H", "M"]

    all_colors = []

    for letter in lst:
        print(f"正在解析 {letter} 系列颜色...")

        try:
            with open(f"input_{letter}.html", "r", encoding="utf-8") as f:
                html = f.read()

            colors = parse_html_to_colors(html)
            print(f"  → 解析到 {len(colors)} 个颜色")

            all_colors.extend(colors)

        except FileNotFoundError:
            print(f"  ⚠️ 未找到 input_{letter}.html，跳过")

    save_to_json(all_colors)

    print(f"\n✅ 已生成 {len(all_colors)} 个颜色 -> artkal_colors.json")

正在解析 A 系列颜色...
  → 解析到 26 个颜色
正在解析 B 系列颜色...
  → 解析到 32 个颜色
正在解析 C 系列颜色...
  → 解析到 29 个颜色
正在解析 D 系列颜色...
  → 解析到 26 个颜色
正在解析 E 系列颜色...
  → 解析到 24 个颜色
正在解析 F 系列颜色...
  → 解析到 25 个颜色
正在解析 G 系列颜色...
  → 解析到 21 个颜色
正在解析 H 系列颜色...
  → 解析到 23 个颜色
正在解析 M 系列颜色...
  → 解析到 15 个颜色

✅ 已生成 221 个颜色 -> artkal_colors.json
